In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, MaxPool2D, Dropout, ZeroPadding2D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import fashion_mnist

2026-07-21 11:58:00.520431: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/lucifer666/anaconda3/envs/py312/lib/python3.12/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


In [2]:
def load_mnist(path, kind='train'):
    labels_path = os.path.join(path,'%s-labels-idx1-ubyte' % kind)
    images_path = os.path.join(path,'%s-images-idx3-ubyte' % kind)

    with open(labels_path, 'rb') as lbpath:
        labels = np.frombuffer(lbpath.read(), dtype=np.uint8, offset=8)
    
    with open(images_path, 'rb') as imgpath:
        images = np.frombuffer(imgpath.read(), dtype=np.uint8, offset=16).reshape(len(labels), 784)
    
    return images, labels

In [3]:
X_train, y_train = load_mnist('.', 'train')
X_test, y_test = load_mnist('.', 't10k')

In [4]:
X_train.shape, y_train.shape

((60000, 784), (60000,))

In [5]:
X_train = X_train.reshape(X_train.shape[0], 28, 28)
X_test = X_test.reshape(X_test.shape[0], 28, 28)

## for convolutional neural networks we need to define the channels of the photo as a dimention

In [6]:
X_train = np.expand_dims(X_train, axis=3)
X_test = np.expand_dims(X_test, axis=3)

In [7]:
X_train.shape, X_test.shape

((60000, 28, 28, 1), (10000, 28, 28, 1))

In [8]:
X_train = X_train / 255
X_test = X_test / 255

In [9]:
model = Sequential([
    ZeroPadding2D(padding=(1, 1), input_shape=(28, 28, 1)),
    Conv2D(32, 3, activation='relu'),
    Dropout(0.2),
    MaxPool2D(pool_size=2, strides=2),
    ZeroPadding2D(padding=(1, 1)),
    Conv2D(64, 3, activation='relu'),
    Dropout(0.2),
    MaxPool2D(pool_size=2, strides=2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')  
])

/home/lucifer666/anaconda3/envs/py312/lib/python3.12/site-packages/keras/src/layers/reshaping/zero_padding2d.py:72: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
2026-07-21 11:58:35.164415: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ zero_padding2d (ZeroPadding2D)  │ (None, 30, 30, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 28, 28, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_1                │ (None, 16, 16, 32)     │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,642 (1.61 MB)

 Trainable params: 421,642 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(
    optimizer= tf.keras.optimizers.Adam(learning_rate=0.001),
    loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics= [tf.keras.metrics.SparseCategoricalAccuracy()]
)
# or ---> model.compile('adam', loss='categorical_crossentropy', metrics= ['accuracy'])

In [ ]:
#y_train_ohe = to_categorical(y_train)
#y_test_ohe = to_categorical(y_test)

In [12]:
history = model.fit(X_train, y_train, batch_size=300, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5


/home/lucifer666/anaconda3/envs/py312/lib/python3.12/site-packages/keras/src/backend/tensorflow/nn.py:717: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


200/200 ━━━━━━━━━━━━━━━━━━━━ 63s 304ms/step - loss: 0.6554 - sparse_categorical_accuracy: 0.7645 - val_loss: 0.4426 - val_sparse_categorical_accuracy: 0.8484
Epoch 2/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 60s 297ms/step - loss: 0.4118 - sparse_categorical_accuracy: 0.8514 - val_loss: 0.3767 - val_sparse_categorical_accuracy: 0.8697
Epoch 3/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 58s 292ms/step - loss: 0.3589 - sparse_categorical_accuracy: 0.8702 - val_loss: 0.3348 - val_sparse_categorical_accuracy: 0.8842
Epoch 4/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 55s 277ms/step - loss: 0.3272 - sparse_categorical_accuracy: 0.8834 - val_loss: 0.3114 - val_sparse_categorical_accuracy: 0.8916
Epoch 5/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 58s 290ms/step - loss: 0.3058 - sparse_categorical_accuracy: 0.8904 - val_loss: 0.2941 - val_sparse_categorical_accuracy: 0.8970


In [14]:
history.history

{'loss': [0.6554405093193054,
  0.41184377670288086,
  0.3588799834251404,
  0.3272295892238617,
  0.3058217763900757],
 'sparse_categorical_accuracy': [0.7644833326339722,
  0.8514000177383423,
  0.8702166676521301,
  0.8834166526794434,
  0.89041668176651],
 'val_loss': [0.4425700604915619,
  0.37668558955192566,
  0.334826797246933,
  0.3113633692264557,
  0.2940615117549896],
 'val_sparse_categorical_accuracy': [0.8483999967575073,
  0.869700014591217,
  0.8841999769210815,
  0.8916000127792358,
  0.8970000147819519]}

In [15]:
model.save('./model.h5')